In [ ]:
# import torch
# import torch.nn as nn

# # Define the atrous convolution layer function
# def atrous_conv_layer(in_channels, out_channels, kernel_size=3, dilation=1):
#     padding = dilation if kernel_size == 3 else 0
#     atrous_conv = nn.Conv2d(in_channels, out_channels, kernel_size, 
#                             padding=padding, dilation=dilation, bias=False)
#     batch_norm = nn.BatchNorm2d(out_channels)
#     relu_actv = nn.ReLU(inplace=True)
    
#     # Return the layers as a sequential container
#     return nn.Sequential(atrous_conv, batch_norm, relu_actv)

# # Instantiate the layers
# branch1 = atrous_conv_layer(3, 64, kernel_size=3, dilation=1)
# branch2 = atrous_conv_layer(3, 64, kernel_size=3, dilation=2)
# branch3 = atrous_conv_layer(3, 64, kernel_size=3, dilation=4)
# branch4 = atrous_conv_layer(3, 64, kernel_size=3, dilation=8)

# # Define a function to pass input through each layer and combine outputs
# def forward_parallel(input_tensor):
#     out1 = branch1(input_tensor)  # Pass input through each branch separately
#     out2 = branch2(input_tensor)
#     out3 = branch3(input_tensor)
#     out4 = branch4(input_tensor)
    
#     # Concatenate along the channel dimension
#     combined = torch.cat([out1, out2, out3, out4], dim=1)
    
#     # Final classification layer
#     final_conv = nn.Conv2d(256, 10, kernel_size=1)  # Adjust to the number of desired output channels
#     return final_conv(combined)

# # Pass input through the network
# input_tensor = torch.randn(1, 3, 224, 224)  # Example input (batch size 1, 3 channels, 224x224)
# output = forward_parallel(input_tensor)

# print("Output shape:", output.shape)  # Expected output shape: [1, 10, height, width]


In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
import torchfit

In [17]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, concatenate, Conv2DTranspose, BatchNormalization, Dropout, Activation, LayerNormalization, MultiHeadAttention, Add, Dense, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import models
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import load_model

In [13]:
batch_size = 32
input_tensor = torch.zeros((batch_size, 3, 256, 256))

In [6]:
# atrous convolutional layer architecture
def atrous_conv_layer(in_channels, out_channels, dilation, kernel_size=3):

    padding = dilation if kernel_size == 3 else 0
    atrous_conv = nn.Conv2d(in_channels, out_channels, kernel_size, 
                                padding=padding, dilation=dilation, bias=False)
    batch_norm = nn.BatchNorm2d(out_channels)
    relu_actv = nn.ReLU(inplace=True)

    # returns a layer (structure)
    return nn.Sequential(atrous_conv, batch_norm, relu_actv)

In [10]:
# global pooling architecture
def global_pooling_layer(in_channels, out_channels):
    
    global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
    conv1x1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
    # apply batch normalisation here if needed
    
    return nn.Sequential(global_avg_pool, conv1x1)

In [ ]:
# Fix this: define out_channels and add batch norm and relu
# final_layer = nn.Conv2d(out_channels *5, out_channels, kernel_size=1, bias='False')

In [12]:
# Atrous Spatial Pyramid pool function ---> parallel convolution + global pooling + 1x1 conv + concat 
def ASPP_multiple_conv(input_tensor, final_layer):

    branch1 = atrous_conv_layer(3, 64, 1, kernel_size=3)(input_tensor)
    branch2 = atrous_conv_layer(3, 64, 2, kernel_size=3)(input_tensor)
    branch3 = atrous_conv_layer(3, 64, 3, kernel_size=3)(input_tensor)
    branch4 = atrous_conv_layer(3, 64, 4, kernel_size=3)(input_tensor)

    global_pooling = global_pooling_layer(3, 64)(input_tensor)

    # Upsampling is done here
    global_avg_pool = nn.Upsample(size=branch1.shape[2:], mode='bilinear', align_corners=True)(global_pooling)

    # concat + 1x1 conv is done here
    concat_output  = torch.cat([branch1, branch2, branch3, branch4, global_avg_pool], dim=1)
    aspp_output = final_layer(concat_output)

    # do we need to pass this output through ReLU?

    return aspp_output

In [14]:
def decoder_architecture(in_channels, num_classes):

    conv1 = nn.Conv2d(in_channels, 256, kernel_size=3, stride=1, padding=1, bias=False)
    conv2 = nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1, bias=False)
    final_conv = nn.Conv2d(256, num_classes, kernel_size=1, stride=1)

    return nn.Sequential(conv1, conv2, final_conv)

In [15]:
def decoder(aspp_output):

    decoded_ouput = decoder_architecture(256, 24)(aspp_output)
    upsampled_decoder_output = nn.Upsample(scale_factor=4, mode='bilinear', align_corners=True)(decoded_ouput)

    return upsampled_decoder_output

In [ ]:
model = 

In [ ]:
model.compile(optimizer=torch.optim.Adam(model.parameters(), lr=1e-4), 
              loss=nn.CrossEntropyLoss())
model.fit(train_loader, val_loader=val_loader, epochs=25)